# 🧠 Pooling Types
**Course:** Deep Learning  
**Lecture:** 3 - Convolutional Neural Networks (CNNs)  
**Part:** b - Pooling Types

---

> **Where we are:** Convolutions extract features, but feature maps get massive. We need downsampling techniques to reduce spatial dimensions and add translation invariance.

**What you'll learn:**
- Max, Average, and Global pooling variants
- Stride-2 convolutions as an alternative to pooling
- Downsampling operations and their pros/cons

**Exam relevance:** ⭐⭐⭐ (Crucial for understanding architecture downsampling mechanisms.)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D, GlobalMaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import mnist

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 📖 Theory: Pooling
Pooling reduces the spatial dimensions (height and width) of the input, aggregating information. 
- **Max Pooling** takes the max value in a window. It acts as a strong feature selector (e.g. "did we find an edge here?").
- **Average Pooling** takes the average. It acts as a smoothing operation.
- **Global Pooling** shrinks the entire HxW dimension into a single 1x1 value per channel.

## 🔀 Your Options
At this step, you can choose between: MaxPool, AvgPool, GAP, GMP, or no pooling (stride-2 conv).
| Option | Pros | Cons | When to use |
|---|---|---|---|
| MaxPooling2D | Preserves strong features, translation invariant | Discards non-max info | Most standard CNNs (VGG, etc.) |
| AveragePooling2D | Preserves background info, smoother | Blurs strong features | Rarely used in spatial downsampling today |
| GlobalAveragePooling2D | Flattens without massive parameter counts | Destroys all spatial info | Right before the classification head |
| GlobalMaxPooling2D | Keeps strongest signal per channel | Ignores context | Sometimes used for feature extraction tasks |
| Stride=2 Conv | Learnable downsampling | Adds parameters | Modern architectures (ResNet, MobileNet) |

In [ ]:
# ✏️ Your Turn: Pooling from scratch
def pool2d_scratch(image, pool_size=2, stride=2, mode='max'):
    # ### YOUR CODE HERE ###
    # 1. Calculate output size
    # 2. Slide a window of size pool_size with given stride
    # 3. If mode=='max', take np.max. If mode=='avg', take np.mean
    pass

# Models definitions (implement small networks)
def build_model(downsample_mode):
    inp = Input(shape=(28, 28, 1))
    x = Conv2D(8, (3, 3), activation='relu', padding='same')(inp)
    
    if downsample_mode == 'max':
        x = MaxPooling2D()(x)
        x = Flatten()(x)
    elif downsample_mode == 'avg':
        x = AveragePooling2D()(x)
        x = Flatten()(x)
    elif downsample_mode == 'gap':
        x = GlobalAveragePooling2D()(x)
    elif downsample_mode == 'stride_conv':
        x = Conv2D(8, (3, 3), strides=2, activation='relu', padding='same')(x)
        x = Flatten()(x)
        
    out = Dense(10, activation='softmax')(x)
    return Model(inp, out, name=f"Model_{downsample_mode}")

In [ ]:
# ✅ Solution
def pool2d_scratch(image, pool_size=2, stride=2, mode='max'):
    h_in, w_in = image.shape
    h_out = (h_in - pool_size) // stride + 1
    w_out = (w_in - pool_size) // stride + 1
    
    output = np.zeros((h_out, w_out))
    
    for i in range(h_out):
        for j in range(w_out):
            patch = image[i*stride : i*stride+pool_size, j*stride : j*stride+pool_size]
            if mode == 'max':
                output[i, j] = np.max(patch)
            else:
                output[i, j] = np.mean(patch)
                
    return output

In [ ]:
# 🔬 Comparison Cell: Pooling Variants
grad = np.linspace(0, 1, 32).reshape(-1, 1) * np.ones((1, 32))
image = grad + np.random.normal(0, 0.1, (32, 32))
image[10:15, 10:25] = 2.0 # Strong feature

out_max = pool2d_scratch(image, pool_size=2, stride=2, mode='max')
out_avg = pool2d_scratch(image, pool_size=2, stride=2, mode='avg')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image, cmap='viridis')
axes[0].set_title(f"Original {image.shape}")

axes[1].imshow(out_max, cmap='viridis')
axes[1].set_title(f"MaxPool {out_max.shape}\n(Preserves strong feature)")

axes[2].imshow(out_avg, cmap='viridis')
axes[2].set_title(f"AvgPool {out_avg.shape}\n(Blurs everything)")

plt.show()

# Train simple models on small subset of MNIST for comparison
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = np.expand_dims(x_train[:5000] / 255.0, -1)
y_train = y_train[:5000]

modes = ['max', 'avg', 'gap', 'stride_conv']
results = {}
for mode in modes:
    model = build_model(mode)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f"Training {mode}...")
    h = model.fit(x_train, y_train, epochs=2, verbose=0, validation_split=0.2)
    results[mode] = h.history['val_accuracy'][-1]

import pandas as pd
display(pd.DataFrame(list(results.items()), columns=['Downsample Mode', 'Val Accuracy']))

## 🎓 Exam Corner

### Question 1 (Practical)
What is the output shape of GlobalAveragePooling2D applied to a (batch, 7, 7, 512) tensor?

<details><summary>💡 Answer</summary>

(batch, 512). The spatial dimensions (7x7) are averaged out.

</details>

### Question 2 (Conceptual)
Why is GlobalAveragePooling often used just before the classification head in modern architectures?

<details><summary>💡 Answer</summary>

It drastically reduces the number of parameters compared to flattening and using dense layers (like in VGG). E.g., Flattening 7x7x512 gives 25088 values, leading to millions of weights in the dense layer. GAP collapses it to 512, saving memory, reducing overfitting, and making the network robust to input size changes.

</details>

### Question 3 (Conceptual)
What is the difference between MaxPool and stride-2 convolution for downsampling?

<details><summary>💡 Answer</summary>

MaxPool is a fixed, non-learnable operation that takes the maximum value in a window. Stride-2 convolution uses learnable weights to perform downsampling, allowing the network to learn *how* to downsample. Modern architectures favor stride-2 convs at the cost of more parameters.

</details>